In [1]:
import pandas as pd
from datetime import datetime
import pytz
import time
from io import BytesIO
import numpy as np
import os
import warnings
pd.set_option('display.max_columns', None)

In [24]:
archivo = "C:/datos/parte_2.csv"
lineas_por_archivo = 500000

with open(archivo, "r", encoding="latin-1") as f:
    encabezado = f.readline()
    archivo_num = 1
    lineas_actuales = 0
    salida = open(f"C:/datos/parte_2_p-{archivo_num}.csv", "w", encoding="latin-1")
    salida.write(encabezado)

    for linea in f:
        if lineas_actuales >= lineas_por_archivo:
            salida.close()

            archivo_num += 1
            lineas_actuales = 0

            salida = open(f"C:/datos/parte_2_p-{archivo_num}.csv", "w", encoding="latin-1")
            salida.write(encabezado)

        salida.write(linea)
        lineas_actuales += 1

    salida.close()

In [2]:
def limpiar_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce").dt.date  # Convertir a fecha

def extraer_fecha(fecha):
    try:
        fecha_str = fecha.split("_")[2]
        return datetime.strptime(fecha_str, "%Y%m%d").date()
    except Exception:
        return None  # En caso de error

In [14]:
df_errores= pd.read_csv("C:/datos/364 - errores - 18.csv", dtype=str, encoding= 'latin-1', engine='python')
#df_errores= pd.read_csv("C:/datos/364 - errores - 18.csv", sep=';' ,dtype=str, encoding= 'latin-1', engine='python')
df_errores.columns = df_errores.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True)

In [15]:
df_errores.head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,IDELOTE,IDEDET_1,ORIGEN_ERROR,CODIGO_ERROR,DESCRIPCION_ERROR
0,1266885,1266882,21/12/2024 02:11:28 PM,3189,Desgravamen Banco de la Nacion,378636,DESGRAVAMEN BN (PLAN PREMIER BASICO),00000000000000375678,NaN,NaN,731467363,Poliza Nueva,11/04/2023,28/02/2058,SOL,1000,0,0,4.19,4.19,KELLY LUISA,BAÑOS,ALCALDE,19880229,2,45015948,20100030595_0144001_20241202_101.TXT,0000101465000010146501440010000000000000037567...,1266885,731467363,Error canal,1172,LA EDAD DEL ASEGURADO (FEC. FIN VIGENCIA: 2058...
1,1266885,1266882,21/12/2024 02:11:28 PM,3189,Desgravamen Banco de la Nacion,378636,DESGRAVAMEN BN (PLAN PREMIER BASICO),00000000000000372790,NaN,NaN,731457405,Poliza Nueva,06/04/2023,28/02/2038,SOL,5000,0,0,8.9,8.9,LUCY VILMA,ROQUE,VILLANUEVA,19680229,2,29315468,20100030595_0144001_20241202_101.TXT,0000091507000009150701440010000000000000037279...,1266885,731457405,Error canal,1172,LA EDAD DEL ASEGURADO (FEC. FIN VIGENCIA: 2038...
2,1266886,1266884,21/12/2024 02:15:02 PM,3189,Desgravamen Banco de la Nacion,378636,DESGRAVAMEN BN (PLAN PREMIER BASICO),00000000000000347230,NaN,NaN,731470990,Poliza Nueva,15/04/2023,28/02/2038,SOL,10000,0,0,8.9,8.9,GLORIA,CARRILLO,SECLEN,19680229,2,16659652,20100030595_0144001_20241202_102.TXT,0000105091000010509101440010000000000000034723...,1266886,731470990,Error canal,1172,LA EDAD DEL ASEGURADO (FEC. FIN VIGENCIA: 2038...


In [36]:
df_errores['PRODUCTO'].value_counts()

PRODUCTO
SCTR PENSION                     447543
VEHICULAR                        153704
BBVA MULTIRIESGO PYME             56424
WEB VEHICULOS                     52389
PLANES MEDICOS EPS                37099
ONCOLÓGICO INTEGRAL CENCOSUD      29370
Seguro Vida Retorno Interbank     27988
AMI FORMACION LABORAL              1935
AMC COLECTIVA REGULAR                 2
Name: count, dtype: int64

In [37]:
df_unicos = df_errores[['IDEENTIDAD','CODIGO_PRODUCTO','PRODUCTO']].drop_duplicates()

In [38]:
df_unicos.head(10)

,IDEENTIDAD,CODIGO_PRODUCTO,PRODUCTO
0,460,4119,BBVA MULTIRIESGO PYME
91,460,NaN,NaN
103299,747,2309,PLANES MEDICOS EPS
103662,747,NaN,NaN
107127,766,2310,AMC COLECTIVA REGULAR
107129,656,NaN,NaN
107130,656,5002,ONCOLÓGICO INTEGRAL CENCOSUD
137539,464,4,VEHICULAR
138187,464,NaN,NaN
304832,633,2309,PLANES MEDICOS EPS


In [6]:
df_errores['CODIGO_ERROR'].value_counts()

CODIGO_ERROR
1106    17215
999     13432
528       901
1211      901
1824      386
974        82
1172       13
101         7
972         4
1092        2
859         1
Name: count, dtype: int64

In [16]:
df_errores.drop(['IDELOTE','IDEDET_1'], axis=1, inplace=True)
df_errores= df_errores.rename(columns={'DESCRIPCION_ERROR':'DESCRIPCION_ERROR_SAS', 
                                       'CODIGO_ERROR':'IDEERROR', 'IDEENTIDAD':'ENTIDAD'})

In [17]:
df_errores["MONEDA"] = df_errores["MONEDA"].replace(r"^\s*$", None, regex=True)
df_errores["MONEDA"] = df_errores["MONEDA"].where(df_errores["MONEDA"].isin(["USD", "SOL"]), "SIN DATO")
df_errores= df_errores.fillna({'LOTES_ANTERIORES':'', 'ORIGEN_ERROR': 'SIN DATO', 'TIPDOCUMENTO':'SIN DATO'})
df_errores.loc[df_errores['ORIGEN_ERROR'] == 'nan', 'ORIGEN_ERROR'] = 'SIN DATO'
df_errores['SUMA_ASEGURADA'] = df_errores['SUMA_ASEGURADA'].str.replace(',', '.', regex=False)
df_errores['TASA'] = df_errores['TASA'].str.replace(',', '.', regex=False)
df_errores['TASA_RECARGO'] = df_errores['TASA_RECARGO'].str.replace(',', '.', regex=False)
df_errores['PRIMABRUTACAN'] = df_errores['PRIMABRUTACAN'].str.replace(',', '.', regex=False)
df_errores['PRIMANETACAN'] = df_errores['PRIMANETACAN'].str.replace(',', '.', regex=False)

In [18]:
df_errores["LOTES_ANTERIORES"] = df_errores["LOTES_ANTERIORES"].astype(str)
#df_errores['FECHA_CARGA'] = pd.to_datetime(df_errores['FECHA_CARGA'], format="%d/%m/%Y", errors='coerce').dt.date
df_errores['FECHA_CARGA'] = pd.to_datetime(df_errores['FECHA_CARGA'], format="%d/%m/%Y %I:%M:%S %p", errors='coerce').dt.date
df_errores['CODIGO_PRODUCTO'] = df_errores['CODIGO_PRODUCTO'].fillna(0).astype('int')
df_errores["PRODUCTO"] = df_errores["PRODUCTO"].astype(str)
df_errores['CODIGO_PLAN'] = df_errores['CODIGO_PLAN'].fillna(0).astype('int')
#df_errores["NOMBRE_DE_PLAN"] = df_errores["NOMBRE_DE_PLAN"].astype(str)
df_errores["COD_DE_CERTIFICADO"] = df_errores["COD_DE_CERTIFICADO"].astype(str)
df_errores['FECINI_ALTA_CERTIFICADO'] = pd.to_datetime(df_errores['FECINI_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
df_errores['FECFIN_ALTA_CERTIFICADO'] = pd.to_datetime(df_errores['FECFIN_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
df_errores["TIPO_MOVIMIENTO"] = df_errores["TIPO_MOVIMIENTO"].astype(str)
df_errores['FEC__INICIO'] = pd.to_datetime(df_errores['FEC__INICIO'], format='%d/%m/%Y', errors='coerce').dt.date
df_errores['FEC__FIN'] = pd.to_datetime(df_errores['FEC__FIN'], format='%d/%m/%Y', errors='coerce').dt.date
df_errores['MONEDA'] = df_errores['MONEDA'].astype(str)
df_errores['SUMA_ASEGURADA'] = pd.to_numeric(df_errores['SUMA_ASEGURADA'], errors="coerce").astype('float64')
df_errores['TASA'] = pd.to_numeric(df_errores['TASA'], errors="coerce").astype('float64')
df_errores['TASA_RECARGO'] = pd.to_numeric(df_errores['TASA_RECARGO'], errors="coerce").astype('float64')
df_errores['PRIMABRUTACAN'] = pd.to_numeric(df_errores['PRIMABRUTACAN'], errors="coerce").astype('float64')
df_errores['PRIMANETACAN'] = pd.to_numeric(df_errores['PRIMANETACAN'], errors="coerce").astype('float64')
df_errores['NOMCOMPLETO'] = df_errores['NOMCOMPLETO'].astype(str)
df_errores['APEPATERNO'] = df_errores['APEPATERNO'].astype(str)
df_errores['APEMATERNO'] = df_errores['APEMATERNO'].astype(str)
df_errores["FECNACIMIENTO"] = limpiar_fecha(df_errores["FECNACIMIENTO"])
df_errores["TIPDOCUMENTO"] = df_errores["TIPDOCUMENTO"].astype(str)
df_errores['NUMDOCUMENTO'] = df_errores['NUMDOCUMENTO'].astype(str)
df_errores['NOMBRE_DE_ARCHIVO'] = df_errores['NOMBRE_DE_ARCHIVO'].astype(str)
df_errores['LINEA_TRAMA'] = df_errores['LINEA_TRAMA'].astype(str)
df_errores['ORIGEN_ERROR'] = df_errores['ORIGEN_ERROR'].astype(str)
df_errores['IDEERROR'] = df_errores['IDEERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else "")

In [19]:
df_errores= df_errores[df_errores['PRIMABRUTACAN']<30000] 

In [20]:
df_errores['FECHA_TRAMA'] = df_errores['NOMBRE_DE_ARCHIVO'].apply(extraer_fecha)
df_errores['FECHA_TRAMA']= pd.to_datetime(df_errores['FECHA_TRAMA'], errors='coerce').dt.date
df_errores["TIPO_SEGURO"] = df_errores["LINEA_TRAMA"].str[:3]
df_errores["TIPO_MOVIMIENTO_NUM"] = df_errores["LINEA_TRAMA"].str[48]
df_errores["TIPO_REGISTRO"] = df_errores["LINEA_TRAMA"].str[43:45]

In [21]:
df_errores.head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,IDEERROR,DESCRIPCION_ERROR_SAS,FECHA_TRAMA,TIPO_SEGURO,TIPO_MOVIMIENTO_NUM,TIPO_REGISTRO
0,1266885,1266882,2024-12-21,3189,Desgravamen Banco de la Nacion,378636,DESGRAVAMEN BN (PLAN PREMIER BASICO),00000000000000375678,NaT,NaT,731467363,Poliza Nueva,2023-04-11,2058-02-28,SOL,1000.0,0.0,0.0,4.19,4.19,KELLY LUISA,BAÑOS,ALCALDE,1988-02-29,2,45015948,20100030595_0144001_20241202_101.TXT,0000101465000010146501440010000000000000037567...,Error canal,1172,LA EDAD DEL ASEGURADO (FEC. FIN VIGENCIA: 2058...,2024-12-02,000,1,56
1,1266885,1266882,2024-12-21,3189,Desgravamen Banco de la Nacion,378636,DESGRAVAMEN BN (PLAN PREMIER BASICO),00000000000000372790,NaT,NaT,731457405,Poliza Nueva,2023-04-06,2038-02-28,SOL,5000.0,0.0,0.0,8.90,8.90,LUCY VILMA,ROQUE,VILLANUEVA,1968-02-29,2,29315468,20100030595_0144001_20241202_101.TXT,0000091507000009150701440010000000000000037279...,Error canal,1172,LA EDAD DEL ASEGURADO (FEC. FIN VIGENCIA: 2038...,2024-12-02,000,1,27
2,1266886,1266884,2024-12-21,3189,Desgravamen Banco de la Nacion,378636,DESGRAVAMEN BN (PLAN PREMIER BASICO),00000000000000347230,NaT,NaT,731470990,Poliza Nueva,2023-04-15,2038-02-28,SOL,10000.0,0.0,0.0,8.90,8.90,GLORIA,CARRILLO,SECLEN,1968-02-29,2,16659652,20100030595_0144001_20241202_102.TXT,0000105091000010509101440010000000000000034723...,Error canal,1172,LA EDAD DEL ASEGURADO (FEC. FIN VIGENCIA: 2038...,2024-12-02,000,1,72


In [57]:
df_errores['FECINI_ALTA_CERTIFICADO'].value_counts()

Series([], Name: count, dtype: int64)

In [13]:
df_errores.info()

<class 'pandas.DataFrame'>
RangeIndex: 32944 entries, 0 to 32943
Data columns (total 35 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   NRO_LOTE                 32944 non-null  str           
 1   LOTES_ANTERIORES         32944 non-null  str           
 2   FECHA_CARGA              32944 non-null  object        
 3   CODIGO_PRODUCTO          32944 non-null  int64         
 4   PRODUCTO                 31142 non-null  str           
 5   CODIGO_PLAN              32944 non-null  int64         
 6   NOMBRE_DE_PLAN           31142 non-null  str           
 7   COD_DE_CERTIFICADO       31142 non-null  str           
 8   FECINI_ALTA_CERTIFICADO  31121 non-null  datetime64[us]
 9   FECFIN_ALTA_CERTIFICADO  31121 non-null  datetime64[us]
 10  IDEDET                   32944 non-null  str           
 11  TIPO_MOVIMIENTO          32944 non-null  str           
 12  FEC__INICIO              32944 non-null  ob